<a href="https://colab.research.google.com/github/airenas/colab/blob/master/egs/liepa3/ASR/transcribe_zipformer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Introduction

This colab notebooks shows how to use Python APIs of [sherpa-onnx](https://github.com/k2-fsa/sherpa-onnx) for speech recongition.

# Install sherpa-onnx

In [ ]:
! pip install sherpa-onnx huggingface_hub soundfile

# Download model

## Login to HuggingFace

In [ ]:
from huggingface_hub import login

login()
print("\nDONE: login")

## Load model

In [ ]:
from huggingface_hub import snapshot_download
from pathlib import Path
import sherpa_onnx

def download_files(tag: str) -> (str, str):
    repo_dir = snapshot_download(tag)
    repo_path = Path(repo_dir)
    return (repo_path / "model/model.onnx", repo_path / "model/tokens.txt")

# model_tag_hf= "Airenas/liepa3.zipformer-ctc.v01"
# model, tokens = download_files(model_tag_hf)
model, tokens="models/model.onnx", "models/tokens.txt"

recognizer = sherpa_onnx.OfflineRecognizer.from_zipformer_ctc(
            model=model,
            tokens=tokens,
            debug=True,
        )

### Select audio

Select wav mono 16kHz

In [ ]:
from ipywidgets import FileUpload
from IPython.display import display, Audio
import io
import soundfile as sf

uploader = FileUpload(accept='.wav', multiple=False)
display(uploader)


# Transcribe audio

In [ ]:
uploaded_file = uploader.value[0]  
audio_bytes = uploaded_file['content']

samples, sample_rate = sf.read(io.BytesIO(audio_bytes))
display(Audio(samples, rate=sample_rate))

print("Sample rate:", samplerate)

s = recognizer.create_stream()
s.accept_waveform(sample_rate, samples)
recognizer.decode_stream(s)

print(s.result.text)
